# Model Training

In [1]:
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor


from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [ ]:
import pickle
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV

In [3]:
from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, StandardScaler

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import VotingRegressor, RandomForestRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import SGDRegressor

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
import mlflow
from mlflow.client import MlflowClient

In [6]:
df = pd.read_csv('final_df.csv')

## Data Analysis

In [7]:
df.shape

(10324, 6)

In [8]:
df.isnull().sum().sort_values(ascending=False).head(5)

Shipment Mode                360
Line Item Insurance (USD)    287
Pack Price                     0
Line Item Quantity             0
Weight (Kilograms)             0
dtype: int64

In [9]:
df.columns

Index(['Shipment Mode', 'Pack Price', 'Line Item Quantity',
       'Weight (Kilograms)', 'Freight Cost (USD)',
       'Line Item Insurance (USD)'],
      dtype='object')

## Train Test Split

In [10]:
X = df[['Pack Price', 'Line Item Quantity','Weight (Kilograms)', 'Freight Cost (USD)','Shipment Mode']]
y = df['Line Item Insurance (USD)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
X_train.shape, X_test.shape

((8259, 5), (2065, 5))

## Implement Feature Engineering Pipeline

In [12]:
def cost_cols(df):

    df['Weight (Kilograms)'] = df['Weight (Kilograms)'].astype(str).str.extract(r'(\d+\.?\d*)')
    df['Weight (Kilograms)'] = pd.to_numeric(df['Weight (Kilograms)'])

    df['Freight Cost (USD)'] = df['Freight Cost (USD)'].astype(str).str.extract(r'(\d+\.?\d*)')
    df['Freight Cost (USD)'] = pd.to_numeric(df['Freight Cost (USD)'])

    df['Weight (Kilograms)'] = df['Weight (Kilograms)'].fillna(df['Weight (Kilograms)'].mean())
    df['Freight Cost (USD)'] = df['Freight Cost (USD)'].fillna(df['Freight Cost (USD)'].mean())

    return df

In [13]:
def missing(df):

    df['Shipment Mode'] = df['Shipment Mode'].fillna('Unknown')

    return df

In [14]:
with open('fe_pipeline.pkl', 'rb') as f:
    loaded_pipeline = pickle.load(f)

In [15]:
loaded_pipeline

Pipeline(steps=[('functiontransformer-1',
                 FunctionTransformer(func=<function cost_cols at 0x0000014FD11AACA0>)),
                ('functiontransformer-2',
                 FunctionTransformer(func=<function missing at 0x0000014FD11AB6A0>)),
                ('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num_trf',
                                                  Pipeline(steps=[('tranform',
                                                                   PowerTransformer()),
                                                                  ('scaling',
                                                                   StandardScaler())]),
                                                  ['Pack Price',
                                                   'Line Item Quantity',
                                                   'Weight (Kilograms)',
                                                   'Freight Cost (USD)']),
                                                 ('cat_trf',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse=False))]),
                                                  ['Shipment Mode'])]))])

In [16]:
X_train.columns

Index(['Pack Price', 'Line Item Quantity', 'Weight (Kilograms)',
       'Freight Cost (USD)', 'Shipment Mode'],
      dtype='object')

In [17]:
X_train_trf = loaded_pipeline.fit_transform(X_train)
X_test_trf = loaded_pipeline.transform(X_test)

## Transform Target Feature

In [18]:
y_train_reshaped = y_train.values.reshape(-1, 1)
y_test_reshaped = y_test.values.reshape(-1, 1)

imputer = SimpleImputer(strategy='median')

y_train_trf = imputer.fit_transform(y_train_reshaped)
y_test_trf = imputer.transform(y_test_reshaped)

## Model Training

In [19]:
import time
import numpy as np

In [102]:
mlflow.set_experiment('Shipment Insurance Cost Prediction')
mlflow.set_tracking_uri('http://127.0.0.1:5000/')

models = {
    'LinearRegression': LinearRegression(),
    'DecisionTreeRegressor': DecisionTreeRegressor(),
    'XGBRegressor': XGBRegressor()
}

results = {}

n = X_test_trf.shape[0]     
p = X_test_trf.shape[1]     

for name, model in models.items():
    with mlflow.start_run(run_name=name):

        start_time = time.time()
        model.fit(X_train_trf, y_train_trf)
        end_time = time.time()
        y_pred = model.predict(X_test_trf)
        
        training_time = end_time - start_time
        mse = mean_squared_error(y_test_trf, y_pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y_test_trf, y_pred)
        r2 = r2_score(y_test_trf, y_pred)
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

        metrics = {'Training_Time':training_time,
                   'MSE':mse,
                   'RMSE':rmse,
                   'MAE':mae,
                   'R2':r2,
                   'Adj_R2':adj_r2
                    }
        

        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, name)
        mlflow.log_param('model_name',name)

        results[name] = metrics

results_df = pd.DataFrame(results).T
print('Model Performance Metrics:\n')
print(results_df)

2026/04/03 09:15:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/03 09:15:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LinearRegression at: http://127.0.0.1:5000/#/experiments/5/runs/6461e10ac221478a8a7d49ea77e89e1a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


2026/04/03 09:15:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/03 09:15:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run DecisionTreeRegressor at: http://127.0.0.1:5000/#/experiments/5/runs/ee6ba7f781bb49a8b7adfea9dc75882e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


2026/04/03 09:15:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/03 09:15:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBRegressor at: http://127.0.0.1:5000/#/experiments/5/runs/fd2da486860c4b7fa48cafe3d6db52bf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
Model Performance Metrics:

                       Training_Time            MSE        RMSE         MAE  \
LinearRegression            0.024193  100372.740104  316.816572  209.272037   
DecisionTreeRegressor       0.149893   19878.908895  140.992585   47.452465   
XGBRegressor                0.233909   10501.609033  102.477359   37.176360   

                             R2    Adj_R2  
LinearRegression       0.478601  0.476572  
DecisionTreeRegressor  0.896736  0.896335  
XGBRegressor           0.945448  0.945236  


## Register Model in MLflow

In [110]:
registered_model = 'xgb_final_model'
run_id = 'fd2da486860c4b7fa48cafe3d6db52bf'
final_model_name = 'XGBRegressor'

model_uri = f'runs:/{run_id}/{final_model_name}'

mlflow.register_model(model_uri=model_uri, name=registered_model)

Successfully registered model 'xgb_final_model'.
2026/04/03 09:52:23 WARNING mlflow.tracking._model_registry.fluent: Run with id fd2da486860c4b7fa48cafe3d6db52bf has no artifacts at artifact path 'XGBRegressor', registering model based on models:/m-dddd69162eb04b768ae54678100008b7 instead
2026/04/03 09:52:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgb_final_model, version 1
Created version '1' of model 'xgb_final_model'.


<ModelVersion: aliases=[], creation_timestamp=1775190143942, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1775190143942, metrics=None, model_id=None, name='xgb_final_model', params=None, run_id='fd2da486860c4b7fa48cafe3d6db52bf', run_link='', source='models:/m-dddd69162eb04b768ae54678100008b7', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

## Load Best Model

In [111]:
registered_model = 'xgb_final_model'
model_version = 1

model_uri = f'models:/{registered_model}/{model_version}'

loaded_model_xgb = mlflow.sklearn.load_model(model_uri)

In [112]:
loaded_model_xgb

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [113]:
loaded_model_xgb.fit(X_train_trf, y_train_trf)
a = loaded_model_xgb.predict(X_test_trf)
r2_score(y_test_trf,a)

0.9454480079652732

In [114]:
scores = cross_val_score(loaded_model_xgb, X_train_trf, y_train_trf, cv=5, scoring='r2')
scores.mean()

0.9315531050686093

In [95]:
X_train.columns

Index(['Pack Price', 'Line Item Quantity', 'Weight (Kilograms)',
       'Freight Cost (USD)', 'Shipment Mode'],
      dtype='object')

In [115]:
loaded_model_xgb.feature_importances_

array([0.2511704 , 0.6589944 , 0.01907575, 0.01430736, 0.01604771,
       0.02423534, 0.01009343, 0.00607554], dtype=float32)

## Hyperparamter Tuning

In [116]:
xgb = loaded_model_xgb

xgb_param_dist = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.001, 0.01, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

xgb_random = RandomizedSearchCV(estimator=xgb, param_distributions=xgb_param_dist, n_iter=20,
                                scoring='r2', cv=5 , verbose=1, n_jobs=-1, random_state=42)

xgb_random.fit(X_train_trf, y_train_trf)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                                          min_child_weight=None, missing=nan,
                                          monotone_constraints=None,
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 1.0],
                                        'learning_rate': [0.001, 0.01, 0.1,
                                                          0.2],
                                        'max_depth': [3, 5, 7, 10],
                                        'n_estimators': [50, 100, 200, 300],
                                        'subsample': [0.7, 0.8, 1.0]},
                   random_state=42, scoring='r2', verbose=1)

In [117]:
tuned_model = xgb_random.best_estimator_
print('Best Parameters:', xgb_random.best_params_)
print('R2 Score:', r2_score(y_test_trf, tuned_model.predict(X_test_trf)))

Best Parameters: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 0.8}
R2 Score: 0.9462812380962917


## Register Tuned Model

In [118]:
with mlflow.start_run(run_name='XGB_Tuned_Shp_Run'):
    
    mlflow.sklearn.log_model(tuned_model, name='tuned_model')
    
    mlflow.log_params(xgb_random.best_params_)

    y_pred = tuned_model.predict(X_test_trf)

    n = X_test_trf.shape[0]     
    p = X_test_trf.shape[1] 
    
    mse = mean_squared_error(y_test_trf, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_trf, y_pred)
    r2 = r2_score(y_test_trf, y_pred)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    metrics = {   'MSE':mse,
                   'RMSE':rmse,
                   'MAE':mae,
                   'R2':r2,
                   'Adj_R2':adj_r2
                    }
        

    mlflow.log_metrics(metrics)

2026/04/03 09:53:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGB_Tuned_Shp_Run at: http://127.0.0.1:5000/#/experiments/5/runs/a0be55fa21d048e99f43994298c72ebf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


In [119]:
client = MlflowClient()
run_id = 'a0be55fa21d048e99f43994298c72ebf'
registered_model = 'xgb_final_model'

new_model_version = client.create_model_version(
    name=registered_model,                
    source=f'runs:/{run_id}/model', 
    run_id=run_id
)

2026/04/03 09:53:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgb_final_model, version 2


## Save Model

In [120]:
with open('final_model.pkl','wb') as f:
    pickle.dump(tuned_model,f)